In [ ]:
FLEET_V6 = '#!/usr/bin/env python3\n"""RSNA Knee Abnormality Detection: Fast Inference & Efficiency Track Pipeline\nSystematic Visible-Data Teacher Probe, Corpus Parity Auditor & Production Engine\n\nKey architectural features:\n  1. Exact parity with Dread\'s preprocessed corpus:\n     - Canonical 44-slice slot contract: Sagittal FS (12), Sagittal (10), Coronal FS (8), Coronal (6), Axial (8)\n     - Per-slot (per-series) 2% and 98% percentile normalization matching scanner contrasts\n     - Full K_EVAL=94 window center evaluation across the entire 44-slice volume\n  2. Stage-by-stage parity auditor comparing live DICOM decoding against all_vols.npy on 20 studies\n  3. Automated visible-data Teacher Probe validation (58 gold + 300 train studies)\n  4. Diagnostic exception logging (FIRST 5 full tracebacks captured)\n  5. Hard guard: aborts loudly if fallbacks > 5% of studies (prevents 0.500 LB submissions)\n  6. Multi-GPU dual-T4 producer-consumer queue with exact index alignment and ID verification\n"""\nimport os, sys, glob, time, queue, threading, traceback, shutil\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nimport numpy as np\nimport pandas as pd\nimport pydicom\nimport cv2\nfrom pydicom.pixel_data_handlers.util import apply_modality_lut\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nimport timm\n\n# Hardware & CUDA configuration\nos.environ.setdefault("HF_HUB_OFFLINE", "1")\nos.environ.setdefault("TRANSFORMERS_OFFLINE", "1")\ntorch.backends.cudnn.benchmark = True\ntorch.backends.cuda.matmul.allow_tf32 = True\n\nLAB = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA", "Lateral OA",\n       "PF OA", "Effusion", "Synovitis", "Baker\'s", "Contusion", "Fracture"]\n\n# Model & Preprocessing Contract (Frontier 96-slice native 384px contract)\nIMG = 384\nCROP_MM = 140.0\nSLOTS = [\n    ("Sagittal", 1, 26),   # Sagittal Fluid-Sensitive (Fat-Suppressed)\n    ("Sagittal", 0, 22),   # Sagittal Non-Fluid-Sensitive (Proton Density / T1 / T2)\n    ("Coronal",  1, 18),   # Coronal Fluid-Sensitive\n    ("Coronal",  0, 12),   # Coronal Non-Fluid-Sensitive\n    ("Axial",   -1, 18),   # Axial (any fluid-sensitivity preference)\n]\nMAXS = sum(k for _, _, k in SLOTS)  # 96 slices\nSLOT_SIZES = [26, 22, 18, 12, 18]\nSAGITTAL_SLOTS = (0, 1)\n_b_slots = np.cumsum([0] + SLOT_SIZES)\n_s_map = np.searchsorted(_b_slots, np.arange(MAXS), side=\'right\') - 1\n\nK_EVAL = 94     # Evaluates all 94 possible 3-slice window centers across 96 slices\nWIN_CHUNK = 14  # Optimal benchmarked chunk size\n\n# Mode Toggle: Set True for standard LB submissions (Anatomical Mirror TTA gives +0.002 AUC)\nUSE_TTA = True\n\nDICOM_HEADER_TAGS = [\n    \'ImageOrientationPatient\',\n    \'ImagePositionPatient\',\n    \'PixelSpacing\',\n    \'InstanceNumber\'\n]\n\n# Exception diagnostics tracker\n_MAX_EXC_LOGS = 5\n_exc_log_count = 0\n_exc_lock = threading.Lock()\n\ndef record_exception(source, sid, exc):\n    global _exc_log_count\n    with _exc_lock:\n        if _exc_log_count < _MAX_EXC_LOGS:\n            _exc_log_count += 1\n            print(f"\\n{\'!\' * 80}", flush=True)\n            print(f"[DIAGNOSTIC EXCEPTION #{_exc_log_count}] in {source} for study {sid}: {type(exc).__name__}: {exc}", flush=True)\n            traceback.print_exc()\n            print(f"{\'!\' * 80}\\n", flush=True)\n\n\n# ============================================================================\n# Architecture Definition\n# ============================================================================\ndef build_backbone(arch, pretrained=False):\n    hybrid = arch.startswith(("maxvit", "maxxvit", "coatnet", "coat_", "convnext"))\n    is_vit = (not hybrid) and any(k in arch for k in ("vit", "deit", "dinov2", "eva", "beit"))\n    kw = dict(pretrained=pretrained, num_classes=0, in_chans=3)\n    if is_vit:\n        kw.update(global_pool="token", dynamic_img_size=True)\n    else:\n        kw.update(global_pool="avg")\n    return timm.create_model(arch, **kw)\n\n\nclass RaptorClassifier(nn.Module):\n    def __init__(self, backbone, F_dim=1024, n=12, drop=0.2):\n        super().__init__()\n        self.backbone = backbone\n        self.norm = nn.LayerNorm(F_dim)\n        self.att = nn.Sequential(\n            nn.Linear(F_dim, 256),\n            nn.Tanh(),\n            nn.Dropout(drop),\n            nn.Linear(256, n)\n        )\n        self.clsW = nn.Parameter(torch.zeros(n, F_dim))\n        self.clsb = nn.Parameter(torch.zeros(n))\n        nn.init.trunc_normal_(self.clsW, std=0.02)\n        self.n = n\n        self.win_chunk = WIN_CHUNK\n\n    def encode(self, x):\n        B, K = x.shape[:2]\n        flat = x.flatten(0, 1)\n        if self.win_chunk > 0:\n            f = torch.cat([self.backbone(c) for c in flat.split(self.win_chunk, 0)], 0)\n        else:\n            f = self.backbone(flat)\n        return f.view(B, K, -1)\n\n    def head(self, feats):\n        h = self.norm(feats)\n        a = self.att(h)\n        a = torch.softmax(a, dim=1)\n        pooled = torch.einsum("bkn,bkf->bnf", a, h)\n        logits = (pooled * self.clsW).sum(-1) + self.clsb\n        return logits\n\n    def forward(self, x):\n        return self.head(self.encode(x))\n\n\n# ============================================================================\n# Robust Preprocessing & DICOM Decoders (Frontier 96-Slice Native 384px Contract)\n# ============================================================================\ndef _read_header_single(item):\n    sid, f = item if isinstance(item, tuple) else ("unknown", item)\n    try:\n        h = pydicom.dcmread(f, stop_before_pixels=True, specific_tags=DICOM_HEADER_TAGS, force=True)\n        iop = getattr(h, \'ImageOrientationPatient\', None)\n        ipp = getattr(h, \'ImagePositionPatient\', None)\n        if iop is not None and ipp is not None and len(iop) == 6:\n            r = np.array(iop[:3], float)\n            c = np.array(iop[3:], float)\n            n = np.cross(r, c)\n            pos = float(np.dot(np.array(ipp, float), n))\n        else:\n            pos = float(getattr(h, \'InstanceNumber\', 0) or 0)\n        ps = getattr(h, \'PixelSpacing\', None)\n        ps = float(ps[0]) if ps is not None else 0.5\n        return pos, f, ps\n    except Exception as e:\n        record_exception("_read_header_single", sid, e)\n        return 0.0, f, 0.5\n\n\ndef read_px(item):\n    sid, f = item if isinstance(item, tuple) else ("unknown", item)\n    if f is None:\n        return None\n    try:\n        d = pydicom.dcmread(f, force=True)\n        a = apply_modality_lut(d.pixel_array, d).astype(np.float32)\n        if str(getattr(d, \'PhotometricInterpretation\', \'\')) == \'MONOCHROME1\':\n            a = a.max() - a\n        return a\n    except Exception as e:\n        record_exception("read_px", sid, e)\n        return None\n\n\ndef mm_crop_resize(a, ps):\n    h, w = a.shape\n    cpx = int(round(CROP_MM / max(ps, 1e-3)))\n    cpx = max(1, min(cpx, min(h, w)))\n    y0 = (h - cpx) // 2\n    x0 = (w - cpx) // 2\n    a = a[y0 : y0 + cpx, x0 : x0 + cpx]\n    return cv2.resize(a, (IMG, IMG), interpolation=cv2.INTER_AREA)\n\n\ndef _safe_fluid(val):\n    if val is None or pd.isna(val):\n        return -1\n    try:\n        return int(float(val))\n    except (ValueError, TypeError):\n        return -1\n\n\ndef pick_series_for_slot(rows, plane, fluid, used):\n    plane_norm = str(plane).strip().capitalize()\n    cands = [r for r in rows if str(r.get(\'Anatomical_Plane\', \'\')).strip().capitalize() == plane_norm and str(r.get(\'SeriesInstanceUID\', \'\')) not in used]\n    if fluid in (0, 1):\n        pref = [r for r in cands if _safe_fluid(r.get(\'Fluid_Sensitive\')) == fluid]\n        if pref:\n            return pref[0]\n    return cands[0] if cands else None\n\n\ndef build_study_optimized(sid, ser_records, test_dir, io_pool):\n    """Loads and preprocesses a study matching Dread\'s corpus contract bit-for-bit."""\n    t0 = time.time()\n    t_header, t_pixel, t_prep = 0.0, 0.0, 0.0\n\n    rows = ser_records.get(sid, [])\n    used = set()\n    slot_plans = []\n\n    # Stage 1: Fast header reading & slice sorting across series\n    t_h0 = time.time()\n    for plane, fluid, k in SLOTS:\n        r = pick_series_for_slot(rows, plane, fluid, used)\n        if r is None:\n            slot_plans.append((None, k, 0.5, None))\n            continue\n        used.add(r[\'SeriesInstanceUID\'])\n\n        # Check both nested test_dir/<sid>/<series> and flat test_dir/<series> layouts\n        sdir = os.path.join(test_dir, str(sid), str(r[\'SeriesInstanceUID\']))\n        if not os.path.exists(sdir):\n            sdir = os.path.join(test_dir, str(r[\'SeriesInstanceUID\']))\n        if os.path.isdir(sdir):\n            fs = [os.path.join(sdir, f) for f in os.listdir(sdir) if f.endswith(".dcm")]\n        else:\n            fs = []\n        if not fs:\n            slot_plans.append((None, k, 0.5, r[\'SeriesInstanceUID\']))\n            continue\n\n        header_items = [(sid, f) for f in fs]\n        if len(fs) > 4:\n            recs = list(io_pool.map(_read_header_single, header_items))\n        else:\n            recs = [_read_header_single(item) for item in header_items]\n        recs.sort(key=lambda x: x[0])\n        med_ps = float(np.median([r[2] for r in recs])) if recs else 0.5\n        ordered = [(f, ps) for _, f, ps in recs]\n        slot_plans.append((ordered, k, med_ps, r[\'SeriesInstanceUID\']))\n\n    t_header = time.time() - t_h0\n\n    # Stage 2: Pick target slices per slot\n    t_p0 = time.time()\n    # Group slice requests by slot to preserve PER-SLOT normalization parity\n    slot_slice_requests = []  # list of list of (target_idx, fp, ps)\n\n    cur_idx = 0\n    all_files_to_read = []\n    for ordered, k, med_ps, ser_uid in slot_plans:\n        this_slot_requests = []\n        if ordered is None or not ordered:\n            cur_idx += k\n            slot_slice_requests.append(this_slot_requests)\n            continue\n        n = len(ordered)\n        lo, hi = int(n * 0.06), int(n * 0.94) - 1\n        hi = max(hi, lo)\n        picks = np.linspace(lo, hi, k).round().astype(int) if n > 1 else [0] * k\n\n        for p in picks:\n            if cur_idx >= MAXS: break\n            fp, ps = ordered[min(p, n - 1)]\n            this_slot_requests.append((cur_idx, fp, ps if ps > 0 else med_ps))\n            all_files_to_read.append((sid, fp))\n            cur_idx += 1\n        slot_slice_requests.append(this_slot_requests)\n        if cur_idx >= MAXS: break\n\n    # Parallel pixel decompression\n    if len(all_files_to_read) > 4:\n        all_loaded = list(io_pool.map(read_px, all_files_to_read))\n    else:\n        all_loaded = [read_px(item) for item in all_files_to_read]\n    t_pixel = time.time() - t_p0\n\n    # Map loaded pixel arrays back to each slot\n    t_c0 = time.time()\n    vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n    loaded_iter = iter(all_loaded)\n    total_valid_slices = 0\n\n    # Stage 3: Exact Per-Slot Percentile Normalization & mm-Cropping (Dread Parity)\n    for this_slot_requests in slot_slice_requests:\n        slot_arrs = []\n        for req in this_slot_requests:\n            arr = next(loaded_iter)\n            slot_arrs.append(arr)\n\n        valid_in_slot = [a for a in slot_arrs if a is not None]\n        total_valid_slices += len(valid_in_slot)\n\n        if valid_in_slot:\n            allpx = np.concatenate([a.ravel() for a in valid_in_slot])\n            loq, hiq = np.percentile(allpx, [2.0, 98.0])\n        else:\n            loq, hiq = 0.0, 1.0\n\n        for req, arr in zip(this_slot_requests, slot_arrs):\n            target_idx, _, ps = req\n            if target_idx >= MAXS: break\n            if arr is None: continue\n            aw = np.clip((arr - loq) / (hiq - loq + 1e-6), 0, 1)\n            aw = mm_crop_resize(aw, ps)\n            vol[target_idx] = (aw * 255).astype(np.uint8)\n\n    mask = (vol.reshape(MAXS, -1).sum(1) > 0).astype(np.uint8)\n    t_prep = time.time() - t_c0\n\n    timings = {\n        "header": t_header,\n        "pixel": t_pixel,\n        "prep": t_prep,\n        "total_io": time.time() - t0,\n        "decoded_slices": total_valid_slices,\n        "total_requested": len(all_files_to_read)\n    }\n    return vol, mask, timings\n\n\n# ============================================================================\n# GPU Inference & Full Window Centers (K_EVAL = 42)\n# ============================================================================\ndef mirror_tri(tri_t, cs):\n    is_sag = torch.tensor([_s_map[c] in SAGITTAL_SLOTS for c in cs], device=tri_t.device).view(-1, 1, 1, 1)\n    return torch.where(is_sag, tri_t.flip(1), tri_t.flip(-1))\n\n\n@torch.no_grad()\ndef predict_study(model, vol, mask, res, dev, use_tta=False):\n    t_m0 = time.time()\n    valid = np.where(mask > 0)[0]\n    if len(valid) < 3:\n        valid = np.arange(min(3, MAXS))\n    lo, hi = int(valid.min()), int(valid.max())\n    cs = [c for c in range(lo + 1, hi) if c - 1 >= lo and c + 1 <= hi]\n    if not cs:\n        cs = [max(1, min((lo + hi) // 2, MAXS - 2))]\n    idx = np.linspace(0, len(cs) - 1, min(K_EVAL, len(cs))).round().astype(int)\n    centers = [cs[i] for i in idx]\n\n    wins = []\n    for c in centers:\n        tri = vol[c - 1 : c + 2].astype(np.float32) / 255.0\n        tri_t = torch.from_numpy(tri)[None]\n        if res != IMG:\n            tri_t = F.interpolate(tri_t, size=(res, res), mode=\'bilinear\', align_corners=False)\n        wins.append(tri_t[0])\n    x = torch.stack(wins).to(dev, non_blocking=True)\n\n    with torch.autocast(\'cuda\', dtype=torch.float16):\n        p_plain = torch.sigmoid(model(x[None]).float())[0]\n        if use_tta:\n            x_m = mirror_tri(x, centers)\n            p_mirr = torch.sigmoid(model(x_m[None]).float())[0]\n            p_final = 0.5 * (p_plain + p_mirr)\n        else:\n            p_final = p_plain\n\n    p_np = p_final.detach().cpu().numpy()\n    t_model = time.time() - t_m0\n    return p_np, t_model\n\n\n# ============================================================================\n# Path Discovery & Checkpoints\n# ============================================================================\ndef find_comp_source():\n    cands = [\n        os.environ.get("RSNA_COMP_ROOT", ""),\n        "/kaggle/input/rsna-knee-abnormality-detection",\n        "/kaggle/input/competitions/rsna-knee-abnormality-detection"\n    ]\n    for c in cands:\n        if c and os.path.exists(os.path.join(c, "train.csv")) and os.path.exists(os.path.join(c, "train_series")):\n            return Path(c)\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if "train.csv" in files and "train_series" in dirs:\n            return Path(root)\n    return None\n\n\ndef find_corpus_source():\n    cands = [\n        "/kaggle/input/knee-raptor-corpus",\n        "/kaggle/input/datasets/dreaddevelopment/knee-raptor-corpus",\n        "ds/corpus1",\n        "/home/s05814802723/projects/knee/ds/corpus1"\n    ]\n    for c in cands:\n        if os.path.exists(os.path.join(c, "all_vols.npy")):\n            return Path(c)\n    for root, dirs, files in os.walk("/kaggle/input"):\n        if "all_vols.npy" in files:\n            return Path(root)\n    return None\n\n\ndef find_test_data():\n    comp_roots = []\n    if "RSNA_COMP_ROOT" in os.environ and os.path.exists(os.environ["RSNA_COMP_ROOT"]):\n        comp_roots.append(os.environ["RSNA_COMP_ROOT"])\n    comp_roots.extend([\n        "/kaggle/input/rsna-knee-abnormality-detection",\n        "/kaggle/input/competitions/rsna-knee-abnormality-detection"\n    ])\n    for r in comp_roots:\n        if not r or not os.path.exists(r):\n            continue\n        test_series_csv = os.path.join(r, "test_series.csv")\n        sample_sub_csv = os.path.join(r, "sample_submission.csv")\n        if not os.path.exists(test_series_csv):\n            continue\n        for cand in ["test_series", "test", "test_images"]:\n            p = os.path.join(r, cand)\n            if os.path.exists(p) and os.path.isdir(p):\n                print(f"[data] Found test image directory: {p}", flush=True)\n                return p, test_series_csv, sample_sub_csv\n    raise FileNotFoundError(f"Could not find competition test data. Checked: {comp_roots}")\n\n\ndef find_checkpoint(prefer_swa=True):\n    if os.environ.get("RAPTOR_CKPT"):  # base954_fleet: pick the seed from the notebook\n        return os.environ["RAPTOR_CKPT"]\n    direct_candidates = [\n        "/kaggle/input/knee-our-models/raptor_ft_v6_swa.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v6_swa.pt",\n        "raptor_ft_v6_swa.pt",\n        "/kaggle/input/knee-our-models/raptor_ft_v6.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v6.pt",\n        "raptor_ft_v6.pt",\n        "/kaggle/input/knee-our-models/raptor_ft_v5s2_swa.pt",\n        "/kaggle/input/datasets/shivoo29/knee-our-models/raptor_ft_v5s2_swa.pt",\n        "raptor_ft_v5s2_swa.pt"\n    ]\n    for p in direct_candidates:\n        if os.path.exists(p):\n            print(f"[ckpt] Found direct checkpoint: {p}", flush=True)\n            return p\n\n    for root in ["/kaggle/input/knee-our-models", "/kaggle/input/datasets/shivoo29/knee-our-models", "."]:\n        if not os.path.exists(root): continue\n        target_fnames = ["raptor_ft_v6_swa.pt", "raptor_ft_v6.pt", "raptor_ft_v5s2_swa.pt", "raptor_ft_v5s2.pt"]\n        for fname in target_fnames:\n            p = os.path.join(root, fname)\n            if os.path.exists(p):\n                print(f"[ckpt] Selected checkpoint: {p}", flush=True)\n                return p\n    raise FileNotFoundError("Could not find model checkpoint in direct paths")\n\n\n# ============================================================================\n# Stage-by-Stage Corpus Parity Auditor (~20 Studies)\n# ============================================================================\ndef audit_corpus_parity(corpus_dir, comp_src, model, res, device, n_check=20):\n    print("\\n" + "=" * 80)\n    print(f"STAGE-BY-STAGE PARITY AUDIT: LIVE DICOM vs ALL_VOLS.NPY ({n_check} Studies)")\n    print("=" * 80)\n\n    c_ids = np.load(corpus_dir / "all_ids.npy", allow_pickle=True).astype(str)\n    c_vols = np.load(corpus_dir / "all_vols.npy", mmap_mode="r")\n    c_masks = np.load(corpus_dir / "all_masks.npy")\n    id2row = {sid: i for i, sid in enumerate(c_ids)}\n\n    ser_csv = comp_src / "train_series.csv"\n    ser_df = pd.read_csv(ser_csv, dtype=str)\n    ser_records = {}\n    for uid, g in ser_df.groupby("StudyInstanceUID"):\n        ser_records[uid] = g.to_dict("records")\n\n    tr = pd.read_csv(comp_src / "train.csv", dtype={"StudyInstanceUID": str})\n    gold_ids = [s for s in tr[tr.ACL.notna()].StudyInstanceUID if s in id2row][:n_check]\n\n    io_pool = ThreadPoolExecutor(max_workers=8)\n    train_series_dir = os.path.join(comp_src, "train_series")\n\n    diff_records = []\n    prob_diffs = []\n\n    for sid in gold_ids:\n        row = id2row[sid]\n        v_corpus = c_vols[row]\n        m_corpus = c_masks[row]\n\n        v_dicom, m_dicom, timings = build_study_optimized(sid, ser_records, train_series_dir, io_pool)\n\n        # Stage metrics\n        max_abs_diff = int(np.abs(v_corpus.astype(int) - v_dicom.astype(int)).max())\n        mean_abs_diff = float(np.abs(v_corpus.astype(float) - v_dicom.astype(float)).mean())\n        mask_match = bool(np.array_equal(m_corpus, m_dicom))\n\n        # Model output difference on both tensors\n        p_corp, _ = predict_study(model, v_corpus, m_corpus, res, device, use_tta=False)\n        p_dcm, _ = predict_study(model, v_dicom, m_dicom, res, device, use_tta=False)\n        max_p_diff = float(np.abs(p_corp - p_dcm).max())\n        cos_sim = float(np.dot(p_corp, p_dcm) / (np.linalg.norm(p_corp) * np.linalg.norm(p_dcm) + 1e-8))\n\n        diff_records.append({\n            "sid": sid[:12] + "...",\n            "max_px_diff": max_abs_diff,\n            "mean_px_diff": mean_abs_diff,\n            "mask_match": mask_match,\n            "max_prob_diff": max_p_diff,\n            "cos_sim": cos_sim\n        })\n        prob_diffs.append(max_p_diff)\n\n    io_pool.shutdown(wait=False)\n\n    df_report = pd.DataFrame(diff_records)\n    print(df_report.to_string(index=False))\n    print("-" * 80)\n    avg_max_p = np.mean(prob_diffs)\n    print(f"Parity Summary: Mean Max Prob Diff: {avg_max_p:.4f} | Mean Cosine Sim: {np.mean([r[\'cos_sim\'] for r in diff_records]):.5f}")\n    if avg_max_p < 0.05:\n        print("[audit] PARITY VERIFIED! DICOM path reproduces corpus tensor representations with high fidelity!")\n    else:\n        print(f"[audit] NOTICE: Slight numerical variance observed ({avg_max_p:.4f} prob diff), verifying on Gold-58 macro-AUC...")\n    print("=" * 80 + "\\n")\n\n\n# ============================================================================\n# Visible-Data Teacher Probe Setup & AUC Evaluation\n# ============================================================================\ndef setup_teacher_probe(comp_src, work_dir="/kaggle/working"):\n    fake = Path(work_dir) / "fakecomp"\n    fake.mkdir(parents=True, exist_ok=True)\n    tr = pd.read_csv(comp_src / "train.csv", dtype={"StudyInstanceUID": str})\n    gold_ids = tr[tr.ACL.notna()].StudyInstanceUID\n    n_sample = min(300, len(tr[tr.ACL.isna()]))\n    sampled_ids = tr[tr.ACL.isna()].StudyInstanceUID.sample(n_sample, random_state=0)\n    all_probe_ids = pd.concat([gold_ids, sampled_ids]).reset_index(drop=True)\n\n    # 1. test.csv\n    pd.DataFrame({"StudyInstanceUID": all_probe_ids}).to_csv(fake / "test.csv", index=False)\n\n    # 2. test_series.csv\n    ts = pd.read_csv(comp_src / "train_series.csv", dtype=str)\n    ts[ts.StudyInstanceUID.isin(all_probe_ids)].to_csv(fake / "test_series.csv", index=False)\n\n    # 3. sample_submission.csv\n    cols = ["StudyInstanceUID"] + LAB\n    pd.DataFrame({c: (all_probe_ids.values if c == "StudyInstanceUID" else 0.5) for c in cols}).to_csv(fake / "sample_submission.csv", index=False)\n\n    # 4. test_series symlink pointing to train_series\n    target_series = fake / "test_series"\n    if not target_series.exists():\n        src_train_series = comp_src / "train_series"\n        target_series.symlink_to(src_train_series)\n\n    print(f"[probe] Built fakecomp at {fake}: {len(gold_ids)} gold + {n_sample} train studies = {len(all_probe_ids)} total", flush=True)\n    return fake, gold_ids.tolist()\n\n\ndef evaluate_gold_auc(sub_df, train_csv_path):\n    from sklearn.metrics import roc_auc_score\n    tr = pd.read_csv(train_csv_path, dtype={\'StudyInstanceUID\': str})\n    gold_df = tr[tr.ACL.notna()].copy()\n    gold_ids = set(gold_df.StudyInstanceUID)\n    matched = sub_df[sub_df.StudyInstanceUID.isin(gold_ids)].copy()\n\n    if len(matched) < len(gold_ids):\n        print(f"[eval] Warning: Only matched {len(matched)} / {len(gold_ids)} gold studies", flush=True)\n    if len(matched) == 0:\n        print("[eval] No gold studies matched in submission. Skipping AUC evaluation.", flush=True)\n        return None\n\n    # Align exactly by StudyInstanceUID\n    gold_df = gold_df[gold_df.StudyInstanceUID.isin(matched.StudyInstanceUID)].copy()\n    gold_df = gold_df.sort_values("StudyInstanceUID").reset_index(drop=True)\n    matched = matched.sort_values("StudyInstanceUID").reset_index(drop=True)\n\n    aucs = {}\n    print("\\n" + "=" * 80)\n    print(f"VISIBLE GOLD-58 VALIDATION RESULTS (K_EVAL={K_EVAL})")\n    print("=" * 80)\n    for col in LAB:\n        y_true = gold_df[col].values.astype(float)\n        y_pred = matched[col].values.astype(float)\n        try:\n            score = roc_auc_score(y_true, y_pred)\n            aucs[col] = score\n            print(f"  {col:<20}: {score:.4f}")\n        except Exception as e:\n            print(f"  {col:<20}: ERR ({e})")\n    macro_auc = float(np.mean(list(aucs.values()))) if aucs else 0.0\n    print("-" * 80)\n    print(f"  MACRO-AUC (Gold 58): {macro_auc:.4f}")\n    print("=" * 80 + "\\n")\n    return macro_auc\n\n\n# ============================================================================\n# Multi-GPU Pipeline Executor\n# ============================================================================\ndef run_pipeline(test_dir, test_series_csv, sample_sub_csv, devices, models, res, is_probe=False):\n    t_start = time.time()\n    tag = "PROBE" if is_probe else "PRODUCTION"\n    print(f"\\n[{tag}] Initializing pipeline run...")\n    print(f"[{tag}] Image dir: {test_dir} | Series CSV: {test_series_csv}")\n\n    ser_df = pd.read_csv(test_series_csv)\n    ser_df["StudyInstanceUID"] = ser_df["StudyInstanceUID"].astype(str)\n    ser_records = {}\n    for uid, g in ser_df.groupby("StudyInstanceUID"):\n        ser_records[uid] = g.to_dict("records")\n\n    sample_sub = pd.read_csv(sample_sub_csv)\n    test_ids = sample_sub["StudyInstanceUID"].astype(str).tolist()\n    n_studies = len(test_ids)\n    print(f"[{tag}] Scoring {n_studies} studies with K_EVAL={K_EVAL}...", flush=True)\n\n    # Shared queues and thread pools\n    io_pool = ThreadPoolExecutor(max_workers=12)\n    work_queue = queue.Queue(maxsize=8)\n    results = [None] * n_studies\n    profile_records = []\n    fallback_count = 0\n    records_lock = threading.Lock()\n    t_steady_start = time.time()\n\n    def gpu_worker(worker_idx, dev, model):\n        nonlocal fallback_count\n        while True:\n            item = work_queue.get()\n            if item is None:\n                work_queue.task_done()\n                break\n            idx, sid, vol, mask, timings = item\n            is_fallback = False\n\n            # Check if volume has zero decoded slices\n            if timings.get("decoded_slices", 0) == 0:\n                is_fallback = True\n                with records_lock:\n                    fallback_count += 1\n                record_exception(f"gpu_worker_{worker_idx}_zero_slices", sid, RuntimeError("Zero valid slices decoded for study!"))\n\n            try:\n                preds, t_model = predict_study(model, vol, mask, res, dev, use_tta=USE_TTA)\n                timings["model"] = t_model\n                timings["worker"] = worker_idx\n            except Exception as e:\n                if not is_fallback:\n                    with records_lock:\n                        fallback_count += 1\n                record_exception(f"gpu_worker_{worker_idx}_predict", sid, e)\n                preds = np.full(12, 0.5, dtype=np.float32)\n                timings["model"] = 0.0\n                timings["worker"] = worker_idx\n\n            row = {"StudyInstanceUID": sid}\n            for j, col in enumerate(LAB):\n                row[col] = float(preds[j])\n            results[idx] = row\n\n            with records_lock:\n                profile_records.append(timings)\n                done_cnt = len(profile_records)\n                if done_cnt % 25 == 0 or done_cnt == n_studies:\n                    cur_steady = time.time() - t_steady_start\n                    print(f"[{tag}] [{done_cnt}/{n_studies}] {cur_steady / done_cnt:.3f}s/study (steady) | {cur_steady:.1f}s elapsed", flush=True)\n            work_queue.task_done()\n\n    # Launch GPU consumer threads\n    consumer_threads = []\n    for w_idx, (dev, m) in enumerate(zip(devices, models)):\n        t = threading.Thread(target=gpu_worker, args=(w_idx, dev, m), daemon=True)\n        t.start()\n        consumer_threads.append(t)\n\n    # Producer: load and preprocess studies in background\n    for idx, sid in enumerate(test_ids):\n        try:\n            vol, mask, timings = build_study_optimized(sid, ser_records, test_dir, io_pool)\n        except Exception as e:\n            fallback_count += 1\n            record_exception("producer_build_study", sid, e)\n            vol = np.zeros((MAXS, IMG, IMG), np.uint8)\n            mask = np.zeros(MAXS, np.uint8)\n            timings = {"header": 0.0, "pixel": 0.0, "prep": 0.0, "total_io": 0.0, "decoded_slices": 0, "total_requested": 0}\n        work_queue.put((idx, sid, vol, mask, timings))\n\n    # Wait for queue completion\n    work_queue.join()\n    for _ in devices:\n        work_queue.put(None)\n    for t in consumer_threads:\n        t.join()\n    io_pool.shutdown(wait=False)\n\n    # Assemble and validate submission DataFrame\n    for idx, r in enumerate(results):\n        if r is None:\n            sid = test_ids[idx]\n            print(f"[FATAL ERROR] results[{idx}] for study {sid} is missing!", flush=True)\n            results[idx] = {"StudyInstanceUID": sid, **{col: 0.5 for col in LAB}}\n\n    sub = pd.DataFrame(results)\n    assert len(sub) == n_studies, f"Row count mismatch: {len(sub)} vs {n_studies}"\n    assert list(sub["StudyInstanceUID"]) == test_ids, "Row ID ordering drift!"\n    sub = sub[["StudyInstanceUID"] + LAB]\n\n    # Metrics\n    steady_time = time.time() - t_steady_start\n    avg_steady_per_study = steady_time / max(1, n_studies)\n    mean_header = np.mean([p["header"] for p in profile_records]) if profile_records else 0\n    mean_pixel = np.mean([p["pixel"] for p in profile_records]) if profile_records else 0\n    mean_prep = np.mean([p["prep"] for p in profile_records]) if profile_records else 0\n    mean_model = np.mean([p.get("model", 0) for p in profile_records]) if profile_records else 0\n    total_decoded_slices = sum(p.get("decoded_slices", 0) for p in profile_records)\n\n    print("\\n" + "-" * 80)\n    print(f"[{tag}] PERFORMANCE PROFILE ({n_studies} Studies)")\n    print("-" * 80)\n    print(f"  • Steady Throughput:       {avg_steady_per_study:.3f} s / study")\n    print(f"  • Total Decoded Slices:    {total_decoded_slices} slices")\n    print(f"  • Header & IPP Sorting:    {mean_header*1000:6.1f} ms")\n    print(f"  • Pixel Decompression:     {mean_pixel*1000:6.1f} ms")\n    print(f"  • Normalization & Crop:    {mean_prep*1000:6.1f} ms")\n    print(f"  • Model Forward:           {mean_model*1000:6.1f} ms")\n    print(f"  • Fallbacks Encountered:   {fallback_count} / {n_studies}")\n    print("-" * 80 + "\\n")\n\n    return sub, profile_records, fallback_count\n\n\n# ============================================================================\n# Main Entry Point\n# ============================================================================\ndef main():\n    t_main_start = time.time()\n    print("=" * 80)\n    print(f"RSNA Knee Inference Pipeline | Mode: {\'FULL (TTA)\' if USE_TTA else \'FAST (No-TTA)\'} | K_EVAL={K_EVAL}")\n    print("=" * 80)\n\n    # Detect hardware\n    num_gpus = torch.cuda.device_count()\n    if num_gpus > 1:\n        devices = [f"cuda:{i}" for i in range(num_gpus)]\n        print(f"[devices] Dual-T4 Multi-GPU enabled: {devices}", flush=True)\n    elif num_gpus == 1:\n        devices = ["cuda:0"]\n        print(f"[devices] Single GPU enabled: {devices[0]}", flush=True)\n    else:\n        devices = ["cpu"]\n        print(f"[devices] CPU only (warning: slow)", flush=True)\n\n    ckpt_path = find_checkpoint(prefer_swa=True)\n    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)\n    arch = ck.get("arch", "coatnet_rmlp_2_rw_384.sw_in12k_ft_in1k")\n    res = int(ck.get("res", 384))\n    print(f"[model] Checkpoint: {ckpt_path} | Arch: {arch} | Res: {res}", flush=True)\n\n    # Initialize model replica per device\n    models = []\n    sample_bb = build_backbone(arch, pretrained=False)\n    f_dim = sample_bb.num_features\n    del sample_bb\n\n    for dev in devices:\n        bb = build_backbone(arch, pretrained=False)\n        m = RaptorClassifier(bb, F_dim=f_dim).to(dev)\n        m.load_state_dict(ck["model"])\n        m.eval()\n        models.append(m)\n\n    # Pre-warm GPU CUDA runtime context and cuDNN autotuner\n    print("[warmup] Pre-warming CUDA runtime & cuDNN autotuner across devices...", flush=True)\n    for dev, m in zip(devices, models):\n        dummy_chunk = torch.zeros((WIN_CHUNK, 3, res, res), dtype=torch.float32, device=dev)\n        dummy_in = dummy_chunk[None]\n        with torch.no_grad(), torch.autocast(\'cuda\', dtype=torch.float16):\n            _ = m(dummy_in)\n        if torch.cuda.is_available():\n            torch.cuda.synchronize(dev)\n    t_startup = time.time() - t_main_start\n    print(f"[startup] Fixed setup & GPU warmup completed in {t_startup:.2f}s", flush=True)\n\n    # Identify real competition test inputs\n    real_test_dir, real_test_series_csv, real_sample_sub_csv = find_test_data()\n    real_sample_sub = pd.read_csv(real_sample_sub_csv)\n    n_real = len(real_sample_sub)\n    print(f"[data] Detected real test set: {n_real} studies from {real_test_dir}", flush=True)\n\n    # Check for competition source datasets to enable Teacher Probe & Parity Auditor\n    comp_src = find_comp_source()\n    corpus_src = find_corpus_source()\n\n    # STAGE 0: CORPUS PARITY AUDIT (if corpus dataset is attached and visible data available)\n    if comp_src is not None and corpus_src is not None and n_real <= 10:\n        try:\n            audit_corpus_parity(corpus_src, comp_src, models[0], res, devices[0], n_check=20)\n        except Exception as e:\n            print(f"[audit] Corpus parity audit exception (non-fatal): {e}", flush=True)\n            traceback.print_exc()\n\n    # STAGE 1: TEACHER PROBE VALIDATION ON VISIBLE DATA\n    should_run_probe = (comp_src is not None) and ((n_real <= 10) or (os.environ.get("RUN_PROBE", "0") == "1"))\n\n    if should_run_probe:\n        print("\\n" + "=" * 80)\n        print("STAGE 1: RUNNING TEACHER PROBE VALIDATION ON VISIBLE DATA")\n        print("=" * 80)\n        probe_root, gold_ids = setup_teacher_probe(comp_src)\n        probe_test_dir = os.path.join(probe_root, "test_series")\n        probe_test_series_csv = os.path.join(probe_root, "test_series.csv")\n        probe_sample_sub_csv = os.path.join(probe_root, "sample_submission.csv")\n\n        probe_sub, probe_records, probe_fallbacks = run_pipeline(\n            probe_test_dir, probe_test_series_csv, probe_sample_sub_csv,\n            devices, models, res, is_probe=True\n        )\n\n        # AUDIT 1: Assert IDs match test.csv and columns match sample_submission.csv\n        probe_test_csv = os.path.join(probe_root, "test.csv")\n        expected_test = pd.read_csv(probe_test_csv, dtype={"StudyInstanceUID": str})\n        assert list(probe_sub["StudyInstanceUID"]) == list(expected_test["StudyInstanceUID"]), \\\n            "FATAL: Probe output StudyInstanceUID order does NOT match test.csv!"\n        assert list(probe_sub.columns) == ["StudyInstanceUID"] + LAB, \\\n            "FATAL: Probe output columns do NOT match sample_submission.csv!"\n\n        # AUDIT 2: Hard guard on fallback count\n        probe_n = len(probe_sub)\n        fallback_rate = probe_fallbacks / max(1, probe_n)\n        print(f"[guard] Probe Fallback Rate: {probe_fallbacks}/{probe_n} ({fallback_rate:.2%})", flush=True)\n        if fallback_rate > 0.05:\n            raise RuntimeError(f"FATAL GUARD TRIGGERED: Probe fallback rate {fallback_rate:.2%} > 5% allowed limit!")\n\n        # AUDIT 3: Validate Gold-58 macro-AUC\n        macro_auc = evaluate_gold_auc(probe_sub, os.path.join(comp_src, "train.csv"))\n        if macro_auc is not None:\n            if macro_auc < 0.85:\n                raise RuntimeError(f"FATAL GUARD TRIGGERED: Gold-58 Macro-AUC is {macro_auc:.4f} < 0.8500! Model predictions are broken/constant!")\n            print(f"[audit] SUCCESS! Visible Gold-58 Macro-AUC: {macro_auc:.4f} (passed >= 0.8500 check)", flush=True)\n    else:\n        print(f"[info] Skipping Teacher Probe (comp_src={comp_src}, n_real={n_real}, RUN_PROBE={os.environ.get(\'RUN_PROBE\')})", flush=True)\n\n    # STAGE 2: PRODUCTION INFERENCE ON REAL TEST SET\n    print("\\n" + "=" * 80)\n    print(f"STAGE 2: SCORING REAL TEST SET ({n_real} Studies)")\n    print("=" * 80)\n    final_sub, final_records, final_fallbacks = run_pipeline(\n        real_test_dir, real_test_series_csv, real_sample_sub_csv,\n        devices, models, res, is_probe=False\n    )\n\n    # Final submission verification\n    assert list(final_sub["StudyInstanceUID"]) == list(real_sample_sub["StudyInstanceUID"]), \\\n        "FATAL: Final submission StudyInstanceUID order does NOT match sample_submission.csv!"\n    assert list(final_sub.columns) == list(real_sample_sub.columns), \\\n        "FATAL: Final submission columns do NOT match sample_submission.csv!"\n    assert not final_sub.isna().any().any(), "FATAL: Final submission contains NaNs!"\n\n    final_fallback_rate = final_fallbacks / max(1, n_real)\n    if final_fallback_rate > 0.05 and n_real > 10:\n        raise RuntimeError(f"FATAL GUARD TRIGGERED: Real test set fallback rate {final_fallback_rate:.2%} > 5% limit!")\n\n    final_sub.to_csv(os.environ.get("RAPTOR_OUTPUT_CSV", "submission.csv"), index=False)\n    print(f"\\n[complete] submission.csv saved successfully with shape {final_sub.shape} and verified valid values!", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n'


In [ ]:

import glob, json, os, subprocess, sys, time
from pathlib import Path
import numpy as np, pandas as pd
NB_T0 = time.time()
WORK = Path('/kaggle/working')
DATA = next(Path(p) for p in ('/kaggle/input/competitions/rsna-knee-abnormality-detection',
                              '/kaggle/input/rsna-knee-abnormality-detection') if (Path(p) / 'test.csv').is_file())
test_ids = pd.read_csv(DATA / 'test.csv')['StudyInstanceUID'].astype(str).tolist()
cols = list(pd.read_csv(DATA / 'sample_submission.csv', nrows=1).columns)
labels = cols[1:]
n = len(test_ids)

def find_one(name):
    hits = glob.glob('/kaggle/input/*/' + name) + glob.glob('/kaggle/input/datasets/*/*/' + name)  # no recursive walk over test DICOMs
    assert len(hits) >= 1, name + ' not attached'
    return hits[0]

def pct(v):
    return pd.Series(np.asarray(v, dtype=np.float64)).rank(method='average', pct=True).values

STOP_AFTER_H = 7.3   # no new student starts after this
SCRIPTS = {'s7': find_one('raptor_ft_v6d_s7_swa.pt'), 's8': find_one('raptor_ft_v6d_s8_swa.pt')}
outs = {}
for tag, ckpt in SCRIPTS.items():
    elapsed_h = (time.time() - NB_T0) / 3600
    if elapsed_h > STOP_AFTER_H:
        print('STUDENT_SKIPPED %s: %.2f h elapsed' % (tag, elapsed_h), flush=True)
        continue
    path = Path('/tmp/student_%s.py' % tag)
    path.write_text(FLEET_V6)
    out = WORK / ('student_%s.csv' % tag)
    env = dict(os.environ, RSNA_COMP_ROOT=str(DATA), RAPTOR_OUTPUT_CSV=str(out), RAPTOR_CKPT=ckpt)
    t = time.time()
    rc = subprocess.run([sys.executable, '-u', str(path)], env=env, cwd='/tmp').returncode
    if rc != 0:
        raise RuntimeError('student %s failed with exit code %d' % (tag, rc))
    print('STUDENT_DONE %s %.0fs' % (tag, time.time() - t), flush=True)
    outs[tag] = pd.read_csv(out, dtype={'StudyInstanceUID': str}).set_index('StudyInstanceUID').loc[test_ids, labels]

assert len(outs) >= 1, 'no student finished'
final = np.column_stack([np.mean([pct(o[c].values) for o in outs.values()], axis=0) for c in labels])
sub = pd.DataFrame(final.astype(np.float32), columns=labels)
sub.insert(0, 'StudyInstanceUID', test_ids)
sub = sub[cols]
assert np.isfinite(sub[labels].values).all() and ((sub[labels].values >= 0) & (sub[labels].values <= 1)).all()
sub.to_csv(WORK / 'submission.csv', index=False)
print('STUDENTS_ONLY_COMPLETE', json.dumps({'students': list(outs), 'rows': n, 'seconds': round(time.time() - NB_T0, 1)}), flush=True)
